<a href="https://colab.research.google.com/github/laug80/25024-Bakend-NodeJS/blob/main/TP_ML_Grupo8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

El objetivo de este trabajo es trabajar con un dataset de ventas online para entender mejor cómo se comportan las compras.



In [131]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [132]:
#Carga del dataset
df=pd.read_csv('sample_data/data.csv', encoding='latin1')
print("Datos cargados")

Datos cargados


In [133]:
#Se imprime la cantidad de datos
print("Filas:" ,f"{df.shape[0]}","Columnas:",f"{df.shape[1]}")

Filas: 541909 Columnas: 8


In [134]:
#Imprimimos las primeras  5 filas y total de columas
pd.options.display.max_columns = None
df.head(5)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [135]:
#Se imprimen el tipo de variables
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [136]:
#Se imprimen las métricas
df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [137]:
#Se buscan nulos
df.isnull().sum()

,0
InvoiceNo,0
StockCode,0
Description,1454
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,135080
Country,0


In [138]:
#Buscamos duplicados
df.duplicated().sum()
duplicados_por_columna = df.apply(lambda col: col.duplicated().sum())

duplicados_por_columna

,0
InvoiceNo,516009
StockCode,537839
Description,537685
Quantity,541187
InvoiceDate,518649
UnitPrice,540279
CustomerID,537536
Country,541871


In [139]:
#Creamos una copia y empezamos a limpiar el dataset.
df_limpio=df.copy()
#Se eliminan solo filas identicas
df_limpio=df.drop_duplicates()
#Se imprime la cantidad de datos que quedan
print("Data set limpio_: Filas:" ,f"{df_limpio.shape[0]}","Columnas:",f"{df_limpio.shape[1]}")

Data set limpio_: Filas: 536641 Columnas: 8


In [140]:
#Antes de eliminar los nulos analizamos si en las demas columas tambien tienen nulos
df[df["Description"].isnull()][["InvoiceNo", "StockCode", "Quantity", "UnitPrice", "CustomerID", "Country"]].head()

,InvoiceNo,StockCode,Quantity,UnitPrice,CustomerID,Country
622,536414,22139,56,0.0,NaN,United Kingdom
1970,536545,21134,1,0.0,NaN,United Kingdom
1971,536546,22145,1,0.0,NaN,United Kingdom
1972,536547,37509,1,0.0,NaN,United Kingdom
1987,536549,85226A,1,0.0,NaN,United Kingdom


Al Tenener columnas como CustomerId y Price sin datos, se consieran filas no útiles

In [141]:
nulos_desc = df[df["Description"].isnull()]

nulos_desc[["Quantity", "UnitPrice", "CustomerID"]].describe()

,Quantity,UnitPrice,CustomerID
count,1454.000000,1454.0,0.0
mean,-9.359697,0.0,NaN
std,243.238758,0.0,NaN
min,-3667.000000,0.0,NaN
25%,-24.000000,0.0,NaN
50%,-3.000000,0.0,NaN
75%,4.000000,0.0,NaN
max,5568.000000,0.0,NaN


In [142]:
#De acuerdo a los resultados anteriores se decide eliminar los nulos
df_limpio = df_limpio.dropna(subset=["Description"])
print("Data set limpio_: Filas:" ,f"{df_limpio.shape[0]}","Columnas:",f"{df_limpio.shape[1]}")

Data set limpio_: Filas: 535187 Columnas: 8


Se decidió utilizar como estratos principales el país y el nivel de volumen de compra, ya que permiten representar distintos mercados y distintos comportamientos de compra.

La variable Quantity se agrupará en tres niveles: bajo, medio y alto, para facilitar la comparación entre tipos de compras.

In [143]:
#Dividimos l columna Queantity (usando quartiles)en 3 niveles par crear grupos equilibrados
df_limpio["Nivel_Volumen"] = pd.qcut(
    df_limpio["Quantity"],
    q=3,
    labels=["Bajo", "Medio", "Alto"],
    duplicates="drop"
)

In [144]:
#Creamos una nueva columna que une Country con Nivel_volumen
df_limpio["Estrato"] = (
    df_limpio["Country"].astype(str) + "_" +
    df_limpio["Nivel_Volumen"].astype(str)
)

In [145]:
# Creamos una tabl par identificar cuantos datos quedaron en los estratos
tabla_estratos = df_limpio["Estrato"].value_counts().reset_index()
tabla_estratos.columns = ["Estrato", "Cantidad"]

tabla_estratos.head(10)

,Estrato,Cantidad
0,United Kingdom_Bajo,228478
1,United Kingdom_Alto,144320
2,United Kingdom_Medio,116048
3,Germany_Alto,5767
4,France_Alto,5189
5,EIRE_Alto,4667
6,Germany_Medio,2450
7,EIRE_Medio,2265
8,France_Medio,2258
9,Netherlands_Alto,2047


In [146]:
#Graficamos

import plotly.express as px

tabla_estratos = df_limpio["Estrato"].value_counts().reset_index()
tabla_estratos.columns = ["Estrato", "Cantidad"]

fig = px.bar(
    tabla_estratos.head(15),
    x="Estrato",
    y="Cantidad",
    title="Cantidad de registros por estrato",
    color="Cantidad",
    color_continuous_scale="RdPu"
)

fig.update_layout(
    plot_bgcolor="#1b1026",
    paper_bgcolor="#1b1026",
    font=dict(color="#F5E6FF"),
    title_font=dict(color="#F5E6FF"),
    xaxis=dict(
        title="Estrato",
        tickangle=-45,
        color="#F5E6FF"
    ),
    yaxis=dict(
        title="Cantidad de registros",
        color="#F5E6FF"
    )

)

fig.show()

Al analizar los estratos se observó un fuerte desbalance, con predominio de United Kingdom. Para evitar estratos con muy pocos registros, se decidió trabajar con los países con mayor cantidad de transacciones, manteniendo una base suficiente para comparar el muestreo aleatorio simple y el muestreo estratificado.

In [147]:
#Se contabilizó la cantidad de transacciones por país y se seleccionaron los cinco países con mayor
#cantidad de registros para trabajar con los estratos más representativos del dataset.
top_paises = df_limpio["Country"].value_counts().head(5).index
df_modelo = df_limpio[df_limpio["Country"].isin(top_paises)].copy()
print(top_paises)

Index(['United Kingdom', 'Germany', 'France', 'EIRE', 'Spain'], dtype='object', name='Country')


In [148]:
#Luego de seleccionar los cinco países con mayor cantidad de transacciones, se volvió a realizar la generación de estratos sobre el nuevo subconjunto de datos. Para ello, se utilizó el país y el nivel de volumen de compra como variables de segmentación, con el objetivo de obtener grupos más representativos y reducir el desbalanceo
#observado en el dataset original.

df_modelo["Nivel_Volumen"] = pd.qcut(
    df_modelo["Quantity"],
    q=3,
    labels=["Bajo", "Medio", "Alto"],
    duplicates="drop"
)

df_modelo["Estrato"] = (
    df_modelo["Country"].astype(str) + "_" +
    df_modelo["Nivel_Volumen"].astype(str)
)

In [149]:
#volovemos a evaluar
df_modelo["Estrato"].value_counts()

,count
Estrato,
United Kingdom_Bajo,228478
United Kingdom_Alto,144320
United Kingdom_Medio,116048
Germany_Alto,5767
France_Alto,5189
EIRE_Alto,4667
Germany_Medio,2450
EIRE_Medio,2265
France_Medio,2258


In [150]:
#Graficamos

tabla_paises = df_modelo["Country"].value_counts().reset_index()
tabla_paises.columns = ["Pais", "Cantidad"]

fig = px.bar(
    tabla_paises,
    x="Pais",
    y="Cantidad",
    title="Cantidad de transacciones por país",
    color="Cantidad",
    color_continuous_scale=["#ff00aa", "#d100ff", "#7a00ff"]
)

fig.update_layout(
    plot_bgcolor="#14001f",
    paper_bgcolor="#14001f",
    font=dict(color="white", size=14),
    title_font=dict(color="#ff4fd8", size=22),
    xaxis=dict(
        title="País",
        tickangle=-45,
        color="white"
    ),
    yaxis=dict(
        title="Cantidad de registros",
        color="white"
    )
)

fig.show()

El dataset está muy concentrado en United Kingdom. Si tomamos una muestra aleatoria simple, existe el riesgo de que los países con menos registros queden poco representados. Por eso se utiliza es conveniente el muestreo estratificado, para asegurar que cada grupo definido tenga presencia dentro de la muestra.

In [151]:
#Creamos una muestra aleatori simple con el 10% del dataset
muestra_simple = df_modelo.sample(
    frac=0.1,
    random_state=22
)

In [152]:
#Ahora usamos los estratos creados anteriormente tambien el 10%
muestra_estratificada = (
    df_modelo
    .groupby("Estrato", group_keys=False)
    .apply(lambda x: x.sample(frac=0.1, random_state=22), include_groups=False)
)

In [153]:
#Comparamos con la tabla país para ver si la muestra mantiene la proporción de países.
comparacion_pais = pd.DataFrame({
    "Dataset original": df_modelo["Country"].value_counts(normalize=True),
    "Muestra simple": muestra_simple["Country"].value_counts(normalize=True),
    "Muestra estratificada": muestra_estratificada["Country"].value_counts(normalize=True)
}) * 100

comparacion_pais

,Dataset original,Muestra simple,Muestra estratificada
Country,,,
United Kingdom,94.448577,94.400866,94.449167
Germany,1.831604,1.850922,1.831601
France,1.650183,1.671239,1.649986
EIRE,1.581208,1.613277,1.580432
Spain,0.488428,0.463696,0.488813


In [154]:
muestra_estratificada["Estrato"] = df_modelo.loc[muestra_estratificada.index, "Estrato"]

In [155]:
comparacion_estratos = pd.DataFrame({
    "Dataset original": df_modelo["Estrato"].value_counts(normalize=True),
    "Muestra simple": muestra_simple["Estrato"].value_counts(normalize=True),
    "Muestra estratificada": muestra_estratificada["Estrato"].value_counts(normalize=True)
}) * 100

comparacion_estratos

,Dataset original,Muestra simple,Muestra estratificada
Estrato,,,
United Kingdom_Bajo,44.143599,44.393137,44.143900
United Kingdom_Alto,27.883666,27.974419,27.883612
United Kingdom_Medio,22.421312,22.033309,22.421655
Germany_Alto,1.114226,1.099347,1.114804
France_Alto,1.002552,1.000811,1.002744
EIRE_Alto,0.901698,0.917733,0.902276
Germany_Medio,0.473358,0.492677,0.473357
EIRE_Medio,0.437614,0.477221,0.436647
France_Medio,0.436262,0.477221,0.436647


Al comparar las distribuciones porcentuales, vemos que la muestra estratificada mantiene prácticamente la misma proporción que el dataset original tanto por país como por estrato. La muestra aleatoria simple también presenta valores similares, aunque con pequeñas variaciones. Esto confirma que el muestreo estratificado permite conservar mejor la representación de los grupos definidos

#Entrenamiento del modelo para clasificación, el objetivo ´redecir el volumen de compras
#Variables predictoras que podemos usar Country UnitPrice y quizás StockCode
# Y entrenamos dos veces: Modelo con muestra aleatoria simple,Modelo con muestra estratificada
#Después comparamos: accuracy, matriz de confusión, classification report

In [182]:
#Entrenamos ambos modelos, nuestra objetivo es predecir el nivel de volumen.
X_simple = muestra_simple[["Country", "UnitPrice"]]
y_simple = muestra_simple["Nivel_Volumen"]

X_train_simple, X_test_simple, y_train_simple, y_test_simple = train_test_split(
    X_simple,
    y_simple,
    test_size=0.2,
    random_state=22,
    stratify=y_simple
)




In [183]:
X_estrat = muestra_estratificada[["Country", "UnitPrice"]]
y_estrat = muestra_estratificada["Nivel_Volumen"]
X_train_estrat, X_test_estrat, y_train_estrat, y_test_estrat = train_test_split(
    X_estrat,
    y_estrat,
    test_size=0.2,
    random_state=42,
    stratify=y_estrat
)

In [184]:
# Usamos OneHotEncoding para convertir Country en variables numéricas
preprocesador = ColumnTransformer(
    transformers=[
        ("country_encoder", OneHotEncoder(handle_unknown="ignore"), ["Country"])
    ],
    remainder="passthrough"
)



In [190]:
modelo_simple = Pipeline(steps=[
    ("preprocesador", preprocesador),
    ("clasificador", DecisionTreeClassifier(random_state=22))
])

In [191]:
modelo_estrat = Pipeline(steps=[
    ("preprocesador", preprocesador),
    ("clasificador", DecisionTreeClassifier(random_state=22))
])

In [192]:
X_train_simple["Country"] = X_train_simple["Country"].astype(str)
X_test_simple["Country"] = X_test_simple["Country"].astype(str)
X_train_estrat["Country"] = X_train_estrat["Country"].astype(str)
X_test_estrat["Country"] = X_test_estrat["Country"].astype(str)

In [193]:
modelo_simple.fit(X_train_simple, y_train_simple)


/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning:


The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).




Pipeline(steps=[('preprocesador',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('country_encoder',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country'])])),
                ('clasificador', DecisionTreeClassifier(random_state=22))])

In [194]:

modelo_estrat.fit(X_train_estrat, y_train_estrat)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning:


The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).




Pipeline(steps=[('preprocesador',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('country_encoder',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country'])])),
                ('clasificador', DecisionTreeClassifier(random_state=22))])

In [195]:
y_pred_simple = modelo_simple.predict(X_test_simple)
y_pred_estrat = modelo_estrat.predict(X_test_estrat)

In [196]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
print("Accuracy muestra simple:")
print(accuracy_score(y_test_simple, y_pred_simple))

print("\nReporte muestra simple:")
print(classification_report(y_test_simple, y_pred_simple))

Accuracy muestra simple:
0.589354714064915

Reporte muestra simple:
              precision    recall  f1-score   support

        Alto       0.55      0.77      0.64      3230
        Bajo       0.68      0.60      0.64      4675
       Medio       0.48      0.33      0.39      2447

    accuracy                           0.59     10352
   macro avg       0.57      0.57      0.56     10352
weighted avg       0.59      0.59      0.58     10352



In [197]:
print("Accuracy muestra estratificada:")
print(accuracy_score(y_test_estrat, y_pred_estrat))

print("\nReporte muestra estratificada:")
print(classification_report(y_test_estrat, y_pred_estrat))

Accuracy muestra estratificada:
0.5931221020092736

Reporte muestra estratificada:
              precision    recall  f1-score   support

        Alto       0.55      0.78      0.64      3222
        Bajo       0.70      0.59      0.64      4654
       Medio       0.47      0.35      0.40      2476

    accuracy                           0.59     10352
   macro avg       0.57      0.57      0.56     10352
weighted avg       0.60      0.59      0.59     10352



In [198]:
from sklearn.metrics import precision_score, recall_score, f1_score

tabla_metricas = pd.DataFrame({
    "Métrica": ["Accuracy", "Precision", "Recall", "F1-Score"],
    "Muestra simple": [
        accuracy_score(y_test_simple, y_pred_simple),
        precision_score(y_test_simple, y_pred_simple, average="weighted"),
        recall_score(y_test_simple, y_pred_simple, average="weighted"),
        f1_score(y_test_simple, y_pred_simple, average="weighted")
    ],
    "Muestra estratificada": [
        accuracy_score(y_test_estrat, y_pred_estrat),
        precision_score(y_test_estrat, y_pred_estrat, average="weighted"),
        recall_score(y_test_estrat, y_pred_estrat, average="weighted"),
        f1_score(y_test_estrat, y_pred_estrat, average="weighted")
    ]
})

tabla_metricas

,Métrica,Muestra simple,Muestra estratificada
0,Accuracy,0.589355,0.593122
1,Precision,0.590877,0.599310
2,Recall,0.589355,0.593122
3,F1-Score,0.580093,0.585719


In [199]:
fig = px.bar(
    tabla_metricas,
    x="Métrica",
    y=["Muestra simple", "Muestra estratificada"],
    barmode="group",
    title="Comparación de métricas entre modelos",
    color_discrete_sequence=["#ff00aa", "#7a00ff"]
)

fig.update_layout(
    plot_bgcolor="#14001f",
    paper_bgcolor="#14001f",
    font=dict(color="white"),
    title_font=dict(color="#ff4fd8")
)

fig.show()

La división de la muestra simple y muestra estratificada mejora levemente las predicciones.

In [200]:
!git status

On branch main
nothing to commit, working tree clean
